# Session 1: momentum is kept, energy may not be
**A. Real-world problem.** Two carts roll toward each other on a smooth track. In one
collision they stick together, in another they bounce apart. What happens to their speeds
afterwards, and what is the same in both collisions?

**Plan.** (1) define momentum and kinetic energy, (2) solve the sticking collision,
(3) solve the perfectly bouncing collision, (4) compare what is conserved.

Educational model: two bodies on a straight line, no friction, no outside forces during
the collision. Not a tool for real crash analysis.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: momentum and kinetic energy
Choose one direction as positive; a body moving the other way has a negative speed.

    p = m v                 momentum (kg m/s)
    KE = 1/2 m v^2          kinetic energy (J)

During the short collision the bodies push on each other with equal and opposite forces, so
the total momentum of the pair does not change: p_before = p_after.

In [ ]:
def momentum(m, v):
    return m * v


def total_momentum(m1, v1, m2, v2):
    return momentum(m1, v1) + momentum(m2, v2)


def kinetic_energy(m, v):
    return 0.5 * m * v ** 2


def total_ke(m1, v1, m2, v2):
    return kinetic_energy(m1, v1) + kinetic_energy(m2, v2)

## Step 2: the two extreme collisions
**Sticking** (perfectly inelastic): the bodies leave together with one common speed. Momentum
conservation gives it directly:

    v = (m1 v1 + m2 v2) / (m1 + m2)

**Bouncing** (perfectly elastic): the kinetic energy is also kept. Momentum and kinetic energy
together give two speeds:

    v1' = ((m1 - m2) v1 + 2 m2 v2) / (m1 + m2)
    v2' = ((m2 - m1) v2 + 2 m1 v1) / (m1 + m2)

In [ ]:
def stick(m1, v1, m2, v2):
    """Common speed after the bodies stick together."""
    if m1 <= 0 or m2 <= 0:
        raise ValueError("masses must be positive")
    return total_momentum(m1, v1, m2, v2) / (m1 + m2)


def bounce(m1, v1, m2, v2):
    """Speeds (v1', v2') after a perfectly elastic collision."""
    if m1 <= 0 or m2 <= 0:
        raise ValueError("masses must be positive")
    return ((m1 - m2) * v1 + 2 * m2 * v2) / (m1 + m2), ((m2 - m1) * v2 + 2 * m1 * v1) / (m1 + m2)


def ke_lost_sticking(m1, v1, m2, v2):
    v = stick(m1, v1, m2, v2)
    return total_ke(m1, v1, m2, v2) - kinetic_energy(m1 + m2, v)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m1, v1, m2, v2 = 2.0, 4.0, 1.0, 0.0
    v = stick(m1, v1, m2, v2)
    a, b = bounce(m1, v1, m2, v2)
    print(f"before: p = {total_momentum(m1, v1, m2, v2):.2f} kg m/s, KE = {total_ke(m1, v1, m2, v2):.2f} J")
    print(f"sticking: v = {v:.3f} m/s, p = {total_momentum(m1, v, m2, v):.2f}, KE = {total_ke(m1, v, m2, v):.2f} J")
    print(f"bouncing: v1' = {a:.3f}, v2' = {b:.3f} m/s, p = {total_momentum(m1, a, m2, b):.2f}, KE = {total_ke(m1, a, m2, b):.2f} J")

    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    labels = ["before", "after sticking", "after bouncing"]
    p = [total_momentum(m1, v1, m2, v2), total_momentum(m1, v, m2, v), total_momentum(m1, a, m2, b)]
    ke = [total_ke(m1, v1, m2, v2), total_ke(m1, v, m2, v), total_ke(m1, a, m2, b)]
    x = np.arange(3)
    ax.bar(x - 0.2, p, 0.4, label="momentum (kg m/s)", color="tab:blue")
    ax.bar(x + 0.2, ke, 0.4, label="kinetic energy (J)", color="tab:orange")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title("Momentum is kept in both; energy only when bouncing")
    ax.grid(alpha=0.3, axis="y")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "conserved.png", dpi=120)

    ratio = np.linspace(0.1, 10, 200)       # m2 / m1, with m1 = 1 kg, v1 = 4 m/s, body 2 at rest
    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    ax2.plot(ratio, [stick(1, 4, r, 0) for r in ratio], color="tab:red", label="sticking: common speed")
    ax2.plot(ratio, [bounce(1, 4, r, 0)[0] for r in ratio], color="tab:blue", label="bouncing: body 1 after")
    ax2.plot(ratio, [bounce(1, 4, r, 0)[1] for r in ratio], color="tab:green", label="bouncing: body 2 after")
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_xlabel("mass ratio m2 / m1 (no unit)")
    ax2.set_ylabel("speed after the collision (m/s)")
    ax2.set_title("A light body bounces back from a heavy one (m1 = 1 kg, v1 = 4 m/s)")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "speeds_vs_mass_ratio.png", dpi=120)

## Where this stops
Real collisions are between sticking and perfectly bouncing. Session 2 adds one number, the
coefficient of restitution, that covers every case in between, and Session 3 leaves the straight line.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
"""Checkpoint for Session 1. Prints PASS or FAIL."""
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    v = L.stick(2, 4, 1, 0)
    checks.append((f"worked example: sticking speed 8 / 3 = 2.667 m/s (got {v:.3f})", abs(v - 8 / 3) < 1e-9))
    a, b = L.bounce(2, 4, 1, 0)
    checks.append((f"worked example: bouncing speeds 1.333 and 5.333 m/s (got {a:.3f} and {b:.3f})", abs(a - 4 / 3) < 1e-9 and abs(b - 16 / 3) < 1e-9))
    checks.append(("worked example: sticking loses 5.33 J of kinetic energy (one third)", abs(L.ke_lost_sticking(2, 4, 1, 0) - 16 / 3) < 1e-9))
    cases = [(2, 4, 1, 0), (3, 2, 1, 0), (1, 5, 4, -2), (5, 0, 2, -3)]
    checks.append(("momentum is kept when sticking", all(abs(L.total_momentum(m1, v1, m2, v2) - L.total_momentum(m1, L.stick(m1, v1, m2, v2), m2, L.stick(m1, v1, m2, v2))) < 1e-9 for m1, v1, m2, v2 in cases)))
    checks.append(("momentum is kept when bouncing", all(abs(L.total_momentum(m1, v1, m2, v2) - (m1 * L.bounce(m1, v1, m2, v2)[0] + m2 * L.bounce(m1, v1, m2, v2)[1])) < 1e-9 for m1, v1, m2, v2 in cases)))
    checks.append(("kinetic energy is kept when bouncing", all(abs(L.total_ke(m1, v1, m2, v2) - L.total_ke(m1, L.bounce(m1, v1, m2, v2)[0], m2, L.bounce(m1, v1, m2, v2)[1])) < 1e-9 for m1, v1, m2, v2 in cases)))
    checks.append(("kinetic energy is never gained when sticking", all(L.ke_lost_sticking(*c) >= -1e-12 for c in cases)))
    a, b = L.bounce(3, 2, 3, 0)
    checks.append(("equal masses and a body at rest: the moving body stops and the other takes its speed", abs(a) < 1e-12 and abs(b - 2) < 1e-12))
    a, b = L.bounce(1, 4, 5, 0)
    checks.append(("a light body hitting a heavy one at rest bounces back", a < 0 < b))
    checks.append(("design: m2 = 8 kg loses at least 75 percent, m2 = 4 kg does not (m1 = 2 kg, v1 = 4 m/s, v2 = 0)",
                   L.ke_lost_sticking(2, 4, 8, 0) / L.total_ke(2, 4, 8, 0) >= 0.75 > L.ke_lost_sticking(2, 4, 4, 0) / L.total_ke(2, 4, 4, 0)))
    checks.append(("edge case: both bodies at rest stay at rest", L.stick(2, 0, 3, 0) == 0 and L.bounce(2, 0, 3, 0) == (0, 0)))
    for bad, label in ((lambda: L.stick(0, 1, 1, 0), "mass 0 when sticking"), (lambda: L.bounce(1, 1, -2, 0), "a negative mass when bouncing")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    return checks
import types
L = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
ok = True
for label, passed in run_checks(L):
    print(('PASS  ' if passed else 'FAIL  ') + label)
    ok &= bool(passed)
print('PASS' if ok else 'FAIL')